__Libraries__

* GeoBrix assumed already installed on the cluster.
* Python bindings now are `databricks.labs.gbx.rasterx` (etc).
* Pandas now at 2.2.3 with DBR17.3
* Match GDAL to version installed with natives (3.11.4).

__Unity Catalog__

* Replace `catalog_name` and `schema_name` with your preferred locations.
* Volume 'data' must exist under `catalog_name`/`schema_name`.

In [0]:
%pip install --quiet shapely folium mapclassify geopandas rasterio==1.4.3 pystac pystac_client planetary_computer tenacity rich

In [0]:
# -- import databricks + delta + spark functions
from delta.tables import *
from pyspark.databricks.sql import functions as DBF
from pyspark.sql import functions as F
from pyspark.sql.functions import col, udf, pandas_udf
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# -- other imports
from datetime import datetime
from io import BytesIO
from matplotlib import pyplot
from rasterio.io import MemoryFile

import geopandas as gpd
import os
import pandas as pd
import pathlib
import planetary_computer
import pyspark
import pystac_client
import requests
import warnings

date_format = "%Y-%m-%d %H:%M:%S"
warnings.simplefilter("ignore")

In [0]:
# https://spark.apache.org/docs/latest/configuration.html
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", 512)

In [0]:
# -- geobrix (assumed on cluster)
from databricks.labs.gbx.rasterx import functions as rx # <- rx.rst_* python bindings

rx.register(spark)             # <- you can invoke functions now via spark sql

In [0]:
catalog_name = "geospatial_docs"
schema_name = "eo_series"

sql(f"""USE CATALOG {catalog_name}""")
sql(f"""CREATE DATABASE IF NOT EXISTS {schema_name}""")
sql(f"""USE DATABASE {schema_name}""")

print(f"... catalog: '{catalog_name}' (USE)")
print(f"... schema: '{schema_name}' (CREATE / USE)")

In [0]:
ETL_DIR = f"/Volumes/{catalog_name}/{schema_name}/data" # <- Volume ('data') must exist
EO_DIR = f"{ETL_DIR}/alaska"
dbutils.fs.mkdirs(EO_DIR)

os.environ['ETL_DIR'] = ETL_DIR
os.environ['EO_DIR'] = EO_DIR

print(f"... ETL_DIR: '{ETL_DIR}'")
print(f"... EO_DIR: '{EO_DIR}' (MKDIRS)")

In [0]:
# -- library.py
import library

In [0]:
%reload_ext autoreload
%autoreload 2
%reload_ext library

In [0]:
# -- 01. Viz Support --

In [0]:
def as_gdf(df, wkt_col = "wkt"):
  gs = gpd.GeoSeries.from_wkt(
    df
    .toPandas()[wkt_col], 
    crs=4326
  )
  pdf = df.drop(wkt_col).toPandas()
  pdf["geometry"] = gs
  gdf = gpd.GeoDataFrame(pdf, geometry="geometry")
  return gdf

In [0]:
def cells_as_gdf(df, cell_col = "cellid", extra_cols=[]):
  cols = [cell_col]
  cols.extend(extra_cols)
  return as_gdf(
    (
      df
        .select(*cols)
        .withColumn("wkt", DBF.h3_boundaryaswkt(cell_col))
    )
  )

In [0]:
# -- 02. Download STACs

In [0]:
@udf(returnType=IntegerType())
def file_size(file_path):
  """
  Return file_size or null.
  - must exist and be a file
  """
  import os

  if not file_path or not isinstance(file_path, (str, bytes, os.PathLike)):
        return None

  if os.path.exists(file_path) and os.path.isfile(file_path):
    return os.path.getsize(file_path)
  else:
    return None


@udf(returnType=StringType())
def timestamp_filename(dt):
  """
  Convert a passed timestamp to a filename friendly output.
  - return looks like 20230131-092030
  """
  from datetime import datetime

  if dt is None:
    return None
  return dt.strftime(library.FILENAME_TIMESTAMP_FORMAT)

def get_now_formatted():
  """
  Use for last update.
  - this is same as used in `timestamp_filename`
  """
  return datetime.now().strftime(library.FILENAME_TIMESTAMP_FORMAT)

def download_band(
    eod_items, band_name, is_append_mode, tbl_prefix="band", eo_dir=EO_DIR, repartition_factor=5,
    do_clean_files=False, do_download=True, do_table_write=True, h3_col = "cellid"
  ):
  """
  Download band into table.
  - sets the 'last_update'
  - assumes databricks catalog and schema already set
  - default is append mode vs overwrite
  - default is 'do_table_write=True'
  - default is 'do_download=True'
  - default is 'do_clean_files=False'
  - filenames are '{band_name}_{item_id}.tif'
  Returns dataframe either from table or generated.
  !!! If you do not write, the returned dataframe will have not yet been executed !!!

  Notes:
  [a] It can take some time to clean files; this will be everything in the <band_name> dir
  [b] It can take some time to download files per band, especially in MPC free tier
  [c] If not doing table write, then the dataframe will not have forced execution on every row,
      you will have to manage that as the caller
  [d] You can change the table prefix to be more isolated for a given analytic / time filter
  """
  _eod_items = eod_items.filter(f"asset.name == '{band_name}'")
  orig_repart_num = spark.conf.get("spark.sql.shuffle.partitions")
  repart_num = round(_eod_items.count() / repartition_factor)
  spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False) # <- option-2: just tweak partition management
  spark.conf.set("spark.sql.shuffle.partitions", repart_num)
  print(f"\t...shuffle partitions to {repart_num} for this operation.")
  try:
    last_updated = get_now_formatted()

    tbl_name = f"{tbl_prefix}_{band_name}"
    eo_dir_band = f"{eo_dir}/{band_name}"

    # [1] what are we downloading?
    #     - see that asset_name is static
    #     - band name will be used below
    to_download = (
      _eod_items
        .repartition(repart_num)
        .groupBy("item_id", "timestamp")
          .agg(
            F.sort_array(F.collect_set(h3_col)).alias("h3_set"),
            *[F.first(cn).alias(cn) for cn in eod_items.columns if cn not in ["item_id", "timestamp", h3_col, "geojson"]]
          )
          .withColumn(
            "band_name",
            F.lit(band_name)
          )
          .withColumn(
            "out_dir_fuse",
            F.lit(eo_dir_band)
          )
          .withColumn(
            "out_filename",
            F.concat(
              col("band_name"), F.lit("_"), col("item_id"),
              F.lit("_"), timestamp_filename("timestamp"), F.lit(".tif")
            )
        )
        .withColumn(
          "last_update",
          F.lit(last_updated)
        )  
    )
    print(f"to_download count? {to_download.count():,}")

    # [3] clean files?
    if do_clean_files:
      dbutils.fs.rm(eo_dir_band, True)

    # [4] do download?
    if do_download:
      to_download = (
        to_download
          .withColumn(
            "out_file_path",
            library.download_asset_v2(
              "item_id",
              "band_name",
              "out_dir_fuse",
              "out_filename"
              )
          )
          .withColumn(
            "out_file_sz",
            file_size("out_file_path")
          )
          .withColumn(
            "is_out_file_valid",
            F
              .when(F.isnull("out_file_sz"), F.lit(False))
              .when(col("out_file_sz") > F.lit(library.FILE_SIZE_THRESHOLD), F.lit(True))
              .otherwise(F.lit(False))
          )
      )
    else:
      to_download = (
        to_download
          .withColumn(
            "out_file_path", 
            F.concat(col("out_dir_fuse"), F.lit("/"), col("out_filename")) # <- path set manually
          )
          .withColumn(
            "out_file_sz",
            F.lit(None).cast("integer") # <- sz unknown
          )
          .withColumn(
            "is_out_file_valid",
            F.lit(None).cast("boolean") # <- validity unknown
          )
      )
  
    # [5] write table?
    # [6] append mode?
    if do_table_write:
      write_mode = "append"
      if not is_append_mode:
        write_mode = "overwrite"
        spark.sql(f"DROP TABLE IF EXISTS {tbl_name}")

      (
        to_download
          .write
            .mode(write_mode)
          .saveAsTable(tbl_name)
      )

      # - return dataframe of written table
      #   !!! this is fully executed !!!
      return spark.table(tbl_name)
    else:
      # - return unwritten dataframe
      #   !!! this is not executed yet !!!
      return to_download
  finally:
    # print(f"...setting shuffle partitions back to {orig_repart_num}")
    spark.conf.set("spark.sql.shuffle.partitions", orig_repart_num)

In [0]:
def update_assets(update_df, band_tbl_name):
  """
  Test the out
  - This expects an existing band tbl_name generated from `download_band(...)`
  - This expects an update_df conforming to what is generated by `download_band(...)`
    to include from `download_missing_assets(...)`
  Returns a dataframe filtered to the merged data (from the band table).
  """
  from datetime import datetime
  
  last_updated = get_now_formatted()
  
  # [1] udf for download_asset
  # [2] re-calc size
  # [3] re-calc valid
  df = (
    update_df
      .drop(
        "last_update",
        "out_file_path", 
        "out_file_sz",
        "is_out_file_valid"
      )
      .withColumn(
        "last_update",
        F.lit(last_updated)
      )
      .withColumn(
        "out_file_path", 
        library.download_asset_v2(
          "item_id",
          "band_name",
          "out_dir_fuse",
          "out_filename"
        )
      )
      .withColumn(
        "out_file_sz",
        file_size("out_file_path")
      )
      .withColumn(
        "is_out_file_valid",
        F
          .when(F.isnull("out_file_sz"), F.lit(False)) # <- null to False
          .when(col("out_file_sz") > F.lit(library.FILE_SIZE_THRESHOLD), F.lit(True))
          .otherwise(F.lit(False))
      )
  )

  # [4] merge changes back to original table
  delta_tbl_eod = DeltaTable.forName(spark, band_tbl_name)

  (
    delta_tbl_eod.alias("eod")
      .merge(
        df.alias("updates"),
        "eod.item_id = updates.item_id"
      ) 
      .whenMatchedUpdate(
        set = {
          "last_update": "updates.last_update",
          "out_file_path": "updates.out_file_path", 
          "out_file_sz": "updates.out_file_sz",
          "is_out_file_valid": "updates.is_out_file_valid"
        }
      )
    .execute()
  ) 
  
  # [5] return the changes
  # - from the 'band_tbl_name'
  return (
    spark.table(band_tbl_name)
      .filter(f"last_update == '{last_updated}'")
  )

def download_missing_assets(
    band_tbl_name, where_clause=None, do_dry_run=False
  ):
  """
  Download missing assets for band (from pre-existing table) and update the table.
  - Columns updated are 'out_file_sz', 'is_out_file_valid',
      and 'last_update'
  - Optional: 'where_clause' can be used to filter the table
  - Missing is based on 'is_out_file_valid' being False or Null
  - Assumes databricks catalog and schema already set
  - default is to update vs dry-run
  Returns the updated or dry run dataframe.
  """
  # - df from table name
  #   filter?
  df = spark.table(band_tbl_name)
  if where_clause is not None:
    df = df.filter(where_clause)
  
  # - filter 'is_out_file_valid' is False or Null
  df = df.filter(
    (col("is_out_file_valid") == False) |
    (F.isnull("is_out_file_valid"))
  )
  
  if not do_dry_run:
    # - handle missing
    return update_assets(df, band_tbl_name)
  else:
    # - just df for dry-run
    #   useful to test 'where_clase'
    return df

In [0]:
# -- 03. Gridded EO Data

In [0]:
def finalize_tiled_band_tbl(
  band_df, tbl_name, repartition_factor = 3,
  item_id_col="item_id", file_col="out_file_path", size_col="size", bbox_col="bbox",
  drop_cols=["timestamp", "h3_set", "asset", "item_properties", "item_collection", "item_bbox", "stac_version", 
        "last_update", "out_dir_fuse", "out_filename", "out_file_path", "out_file_sz", "is_out_file_valid"],
  do_try_open=False, do_display=True, do_overwrite=False, do_append=False
):
  """
  Finalize a tile table for a band dataframe.
  - tile_threshold: size in MB (default 32MB)
  - if you have some bad data, set do_try_open=True
  Returns the resulting dataframe (from the spark table generated).
  """
  _band_df = None
  try:
    if do_overwrite:
      sql(f"""drop table if exists {tbl_name}""")
    if not spark.catalog.tableExists(tbl_name) or do_append:
      _band_df = (
        band_df
          .repartition(round(band_df.count() / repartition_factor))
      )
      _band_df.persist(pyspark.StorageLevel.DISK_ONLY)
      band_cnt = _band_df.count()
      print(f"... number of files to process? {band_cnt:,}")

      if do_try_open:
        _band_df = _band_df.filter(rx.rst_tryopen("tile"))
      (
        _band_df
          .withColumn(size_col, rx.rst_memsize("tile"))
          .withColumn("tile", rx.rst_initnodata("tile"))
          .withColumn(bbox_col, rx.rst_boundingbox("tile"))
          .withColumn("srid", rx.rst_srid("tile"))
        .drop(*drop_cols)
        .write
          .option("mergeSchema", "true")
          .mode("append")
          .saveAsTable(tbl_name)
      )
    sql(f"""ALTER TABLE {tbl_name} CLUSTER BY ({item_id_col})""")
    sql(f"""OPTIMIZE {tbl_name}""")

    tile_df = spark.table(tbl_name)
    if do_display:
      print(f"\n-- {tbl_name}")
      print(f"count? {tile_df.count():,}")
      tile_df.limit(10).display()
    return tile_df
  finally:
    if _band_df is not None:
      _band_df.unpersist()

In [0]:
def gen_tessellate_tiled_band(
  tile_df, h3_res, tbl_name, h3_col="cellid",
  do_try_open=False, do_display=True, do_overwrite=False, do_append=False
):
  """
  Generate H3 tessellation table from a tiled band dataframe.
  - Expects 'tile' column to be present.
  - If you have bad tiffs, set do_try_open=True
  - MLJ: set max_records_per_file, defaults to 1000 (about control over the number of files created)
  Returns the resulting dataframe (from the spark table generated).
  """
  if do_overwrite:
      sql(f"""drop table if exists {tbl_name}""")
  if not spark.catalog.tableExists(tbl_name) or do_append:
    if do_try_open:
      _tile_df = tile_df.filter(rx.rst_tryopen("tile")).filter(F.isnotnull("tile"))
    else:
      _tile_df = tile_df.filter(F.isnotnull("tile"))
    
    write_df = (
      _tile_df
        .sort(F.rand())
        .withColumn("tile", rx.rst_h3_tessellate("tile", F.lit(h3_res)))
        .selectExpr(f"tile.cellid as {h3_col}", "*")
        .write
          .option("mergeSchema", "true")
          .mode("append")
        .saveAsTable(tbl_name)
    )
    sql (f"""ALTER TABLE {tbl_name} CLUSTER BY ({h3_col})""")
    sql(f"""OPTIMIZE {tbl_name}""")

  h3_df = spark.table(tbl_name)
  if do_display:
    print(f"\n-- {tbl_name}")
    print(f"count? {h3_df.count():,}")
    h3_df.limit(10).display()
  return h3_df